# M0 explorer

Inspect one take: waveform, detected click grid, detected note onsets, pitch track and findings.
Use it to understand *why* a take was misjudged before you change any thresholds.

Needs the notebook extra: `pip install -e ".[notebook]"`. Everything runs locally.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import librosa

from coreguitar.analyze import analyze
from coreguitar.audio import load_audio, guitar_band
from coreguitar.config import SR
from coreguitar.reference import load_exercise

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
AUDIO = ROOT / 'data/m0/your_take.wav'          # <- change
EXERCISE = ROOT / 'exercises/chromatic_1234.json'  # <- change, or None for free mode
BPM = None                                         # override the exercise tempo if needed

In [ ]:
ex = load_exercise(EXERCISE, bpm=BPM) if EXERCISE else None
res = analyze(AUDIO, ex, free_bpm=BPM)
y, _, _ = load_audio(AUDIO)
yg = guitar_band(y)
t = np.arange(len(y)) / SR
print('verdict:', res['quality']['verdict'], res['quality']['issues'])
print('click grid:', res['click_grid'])
print('metrics:', res['quality']['metrics'])

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(16, 7), sharex=True)
a1.plot(t, y, lw=0.4, color='0.6', label='raw')
a1.plot(t, yg, lw=0.4, color='C0', label='guitar band')
cg = res['click_grid']
if cg and cg['found']:
    beat = 60 / cg['bpm_measured']
    for k in np.arange(cg['first_click_s'], t[-1], beat):
        a1.axvline(k, color='C2', lw=0.6, alpha=0.5)
for n in res['notes']:
    a1.axvline(n['onset_s'], color='C3', lw=0.8)
    a1.text(n['onset_s'], a1.get_ylim()[1] * 0.9, n['name'] or '?', fontsize=7, color='C3')
for row in res['note_table']:
    if row.get('expected_s') is not None:
        a1.axvline(row['expected_s'], color='k', lw=0.5, ls=':')
a1.set_title('green = clicks, red = detected onsets, dotted = expected (reference)')
a1.legend(loc='upper right')

f0, voiced, _ = librosa.pyin(yg, fmin=70, fmax=1200, sr=SR, hop_length=256)
tt = librosa.times_like(f0, sr=SR, hop_length=256)
a2.plot(tt, librosa.hz_to_midi(f0), '.', ms=2)
for n in res['notes']:
    if n['midi'] is not None:
        a2.hlines(n['midi'], n['onset_s'], n['offset_s'], color='C3')
a2.set_ylabel('MIDI'); a2.set_xlabel('s')
plt.tight_layout(); plt.show()

In [ ]:
for row in res['note_table']:
    print(row)

In [ ]:
for f in res['findings']:
    print(f"{f['id']:>4} {f['type']:<22} conf={f['confidence']:.2f} exp={f['experimental']}  {f['message']}")
print()
for p in res['priorities']:
    print(p['rank'], p['title'], '|', p['observation'], '| drill:', p['drill'])

### Listen to the evidence
Each finding has `evidence.play_from_s` / `play_to_s`. Play that excerpt:

In [ ]:
from IPython.display import Audio
f = res['findings'][0] if res['findings'] else None
if f:
    a, b = f['evidence']['play_from_s'], f['evidence']['play_to_s']
    display(Audio(y[int(a * SR):int(b * SR)], rate=SR))